# Module 2.2: Feature Engineering & Temporal Lag Variables
### 🇵🇱 Inżynieria Cech i Zmienne Opóźnione (Lag Features)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/max480425/industrial-ml-course/blob/main/public/notebooks/m02_l02_feature_engineering_lag_variables.ipynb)

**Industrial Machine Learning & Predictive Maintenance Interactive Laboratory**

> **Summary:** Engineer historical memory into tabular models by creating rolling means, standard deviations, and multi-step lag variables (t-1, t-5, t-30).  
> 🇵🇱 **Streszczenie:** Wprowadź pamięć procesu do modeli tabelarycznych, tworząc kroczące statystyki oraz zmienne opóźnione w czasie (t-1, t-5, t-30).

- **Estimated Time:** 25 minutes
- **Runtime:** Python 3.10+ (CPU or Free Tier T4 GPU)
- **Packages:** `numpy`, `scipy`, `scikit-learn`, `pandas`, `matplotlib`, `seaborn`
---


In [ ]:
# Step 1: Install & Verify Python Scientific Environment
!pip install -q numpy scipy scikit-learn pandas matplotlib seaborn

import numpy as np
import pandas as pd
import scipy.signal as signal
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'monospace'
print("✓ Industrial ML environment loaded successfully.")


### Key Engineering Concepts:
1. **Autoregressive Lag Features**: Static snapshots lack momentum. Lagged variables ($x_{t-1}, x_{t-5}, x_{t-30}$) allow standard tabular models to capture derivatives and acceleration.
2. **Statistical Descriptors**:
   - **RMS (Root Mean Square)**: Measures continuous destructive vibration energy.
   - **Kurtosis**: Measures peakedness/transient impact shocks (sharp bearing spall strikes).
   - **Crest Factor**: Peak amplitude divided by RMS.
3. **Temporal Leakage Prevention**: Rolling statistics must strictly look backwards (`closed='left'` or non-overlapping historic windows) to prevent model predictions from peeking into future timestamps.


In [ ]:
# 1. Synthesize Compressor Vibration with Gradual Bearing Degradation
np.random.seed(42)
n_steps = 1500
t = np.arange(n_steps)

# Baseline healthy vibration
vib = np.random.normal(0, 1.0, n_steps)

# Inject progressive bearing damage starting at step 800 (growing impact pulses)
for idx in range(800, n_steps):
    if np.random.rand() < 0.15:
        severity = (idx - 800) / (n_steps - 800) * 8.0
        vib[idx] += np.random.choice([-1, 1]) * severity

df_vib = pd.DataFrame({"vibration_accel": vib})
print(f"✓ Synthetic bearing vibration stream created ({n_steps} steps).")


In [ ]:
# 2. Extract Rolling RMS, Kurtosis, and Time Lags
from scipy.stats import kurtosis

# Lag features
df_vib["lag_1"] = df_vib["vibration_accel"].shift(1)
df_vib["lag_5"] = df_vib["vibration_accel"].shift(5)

# Rolling RMS (Window = 50 steps)
df_vib["rolling_rms"] = df_vib["vibration_accel"].rolling(50).apply(lambda x: np.sqrt(np.mean(x**2)))

# Rolling Kurtosis (Window = 50 steps)
df_vib["rolling_kurtosis"] = df_vib["vibration_accel"].rolling(50).apply(lambda x: kurtosis(x, fisher=True))

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

axes[0].plot(t, df_vib["vibration_accel"], color='#64748b', alpha=0.6, label='Raw Vibration')
axes[0].axvline(800, color='#ef4444', linestyle='--', label='Incipient Damage Inception')
axes[0].set_ylabel("Accel [g]")
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, df_vib["rolling_rms"], color='#0284c7', linewidth=2, label='Rolling RMS (Energy)')
axes[1].set_ylabel("RMS [g]")
axes[1].legend(loc='upper left')
axes[1].grid(True, alpha=0.3)

axes[2].plot(t, df_vib["rolling_kurtosis"], color='#f59e0b', linewidth=2, label='Rolling Kurtosis (Shock Peakedness)')
axes[2].axhline(0.0, color='gray', linestyle=':', label='Gaussian Normal Baseline (Kurtosis=0)')
axes[2].set_ylabel("Kurtosis")
axes[2].set_xlabel("Time Step")
axes[2].legend(loc='upper left')
axes[2].grid(True, alpha=0.3)

plt.suptitle("Feature Extraction for Mechanical Degradation Tracking", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 📋 Practical Hands-On Checklist

- [ ] **Task 1:** Construct temporal lag features (shift by 1, 5, 15, and 30 minutes) *(🇵🇱 Stwórz cechy opóźnione w czasie (przesunięcia o 1, 5, 15 i 30 minut))*
- [ ] **Task 2:** Compute rolling window statistical aggregations (mean, std, min, max, skew) *(🇵🇱 Oblicz statystyki w oknach przesuwnych (średnia, odchylenie, min, max, skośność))*
- [ ] **Task 3:** Prevent temporal data leakage by dropping forward-looking shift operations *(🇵🇱 Zabezpiecz potok przed wyciekiem danych (data leakage) eliminując przesunięcia w przód)*


In [ ]:
# Exercise Challenge:
# Assert that rolling kurtosis after damage inception (t > 1000) exceeds 3.0
post_damage_kurtosis = df_vib.loc[1000:, "rolling_kurtosis"].mean()
print(f"Average Post-Damage Kurtosis: {post_damage_kurtosis:.2f}")
assert post_damage_kurtosis > 2.5, "Kurtosis failed to register bearing impacts."
print("✓ Bearing feature extraction verified.")
